In [ ]:
# import necessary libraries
import os
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr

In [ ]:
# Load environment variables
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
anthropic_url = os.getenv("anthropic_url")
gemini_url = os.getenv("gemini_url")
ollama_url = os.getenv("ollama_url")        

print("API keys:")
if openai_api_key:
    print("OpenAI API key is set.")
else:
    print("OpenAI API key is NOT set.")
if anthropic_api_key:
    print("Anthropic API key is set.")
else:
    print("Anthropic API key is NOT set.")
if google_api_key:
    print("Google API key is set.")
else:
    print("Google API key is NOT set.") 

print("\nAPI URLs:")
print(f"Anthropic URL: {anthropic_url}")
print(f"Gemini URL: {gemini_url}")
print(f"Ollama URL: {ollama_url}")



API keys:
OpenAI API key is set.
Anthropic API key is set.
Google API key is set.

API URLs:
Anthropic URL: https://api.anthropic.com/v1/
Gemini URL: https://generativelanguage.googleapis.com/v1beta/openai/
Ollama URL: http://localhost:11434/v1/ollama


In [ ]:
# Initialize API clients
openai = OpenAI()
anthropic = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
gemini = OpenAI(base_url=gemini_url, api_key=google_api_key)
ollama = OpenAI(base_url=ollama_url)


In [10]:


system_message = "You are a helpful assistant"

def message_to_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages= messages
    )
    return response.choices[0].message.content

In [11]:
message_to_gpt("What is today's date?")

"Today's date is June 11, 2024."

In [12]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [13]:
shout("hello")

Shout has been called with input hello


'HELLO'

## User Interface time!

In [ ]:
# Local runtime
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [19]:
# Adding share=True means that it can be accessed publically
# A more permanent hosting is available using a platform called Spaces from HuggingFace, which we will touch on next week
# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True. 
# If you're at work on on a work network, I suggest skip this test.


gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True,inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://44caf579da51bd9b0d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Adding authentication

Gradio makes it very easy to have userids and passwords

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [20]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("qamar", "H@shmi@1998"),share=True)

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://d179e09822ff2f5946.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Shout has been called with input Hi


In [23]:
# Adding a little more :

message_input = gr.Textbox(label="Ask me Anything !", info="Enter the question you want to ask", lines=7)
message_output = gr.Textbox(label="Response", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Hello","How are you ?"],
    flagging_mode="never"
    )
view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7868
* Running on public URL: https://cb2e26158b6dd2a952.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Shout has been called with input Hello


In [46]:
# Claude API 
claude_message = "You are a helpful assistant"

def message_to_claude(prompt):
    messages = [
        {"role": "system", "content": claude_message},
        {"role": "user", "content": prompt}
    ]
    stream = anthropic.chat.completions.create(
        model = "claude-sonnet-4-6",
        messages=messages,
        stream=True
    )
    response = ""
    for chunk in stream:
        response +=chunk.choices[0].delta.content or ""
        yield response

In [ ]:
# Adding frontier for above Claude :

message_input = gr.Textbox(label="Ask me Anything !", info="Enter the question you want to ask", lines=7)
message_output = gr.Markdown(label="Response")

view = gr.Interface(
    fn=message_to_claude,
    title="Qamar Chatbot",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Who is PM best PM of India","How to get started as a AI Engineer"],
    flagging_mode="never"
    )
view.launch(share=True,auth=('qamar','mypassword'))

* Running on local URL:  http://127.0.0.1:7877
* Running on public URL: https://7cb2a85edc72891f1e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/Users/mdqamarhashmi/Documents/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mdqamarhashmi/Documents/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mdqamarhashmi/Documents/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mdqamarhashmi/Documents/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^